# Train CVAE for the rolling hour-by-hour backtest (steven4, HORIZON=1)

Run this notebook's kernel connected to a Colab runtime (VS Code: kernel picker top-right -> "Select Another Kernel" -> Google Colab -> pick a GPU runtime).

Separate, focused notebook from `colab_train.ipynb`/`colab_train_generative.ipynb` -- those train the pre-pivot HORIZON=3 checkpoints and stay untouched (their checkpoints can no longer even load against this branch's model code, see `rolling_hour_backtest.md`). This notebook trains the HORIZON=1 CVAE (and, optionally, PatchTST) and runs the new rolling hour-by-hour backtest (`src/rolling_backtest.py`) against it -- predict one candle at a time, re-predict every real hour, hold open-endedly, exit on a divergence loss, a downtrend consensus call, or a 1% hard stop-loss. See `rolling_hour_backtest.md` for the full writeup, including why the trade signal here (a consensus-across-k-samples gate) is a deliberate re-test of something already found net-harmful for the old bracket-order strategy -- read any results here skeptically, not at face value.

Clones/checks out the `steven4` branch specifically. Run the cells top to bottom.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os

REPO_URL = "https://github.com/WoodyChang21/ECE1508_GenAI.git"
BRANCH = "steven4"

if not os.path.isdir("ECE1508_GenAI"):
    !git clone -b {BRANCH} {REPO_URL}
else:
    # fetch + hard reset, not a plain pull -- see colab_train.ipynb's clone cell for why
    # (a pull can fail outright on local changes and Colab just prints the error and
    # moves on, leaving training to silently proceed on stale code).
    !cd ECE1508_GenAI && git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}

%cd ECE1508_GenAI
!git log --oneline -1

In [ ]:
# torch is preinstalled on Colab; just need mplfinance + pyyaml
!pip install -q mplfinance pyyaml

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Sanity checks (full test suite, including the new rolling-backtest tests)

Cheap to run first -- confirms the HORIZON=1 migration (data pipeline, losses, model shapes) and the new `rolling_backtest.py` state machine before committing to a long training run.

In [ ]:
!pip install -q pytest
!python -m pytest steven/tests/ -v

## Momentum feature setup (EMA9/EMA21 + RSI-14 + VIX)

`configs/cvae_h1.yaml` trains on the base hourly OHLCV plus three added features (`src/momentum_pipeline.py`) -- EMA9/EMA21 crossover, RSI-14, and VIX (previous trading day's close). VIX isn't in the hourly parquet, so it's pulled fresh here via `yfinance` before training -- same convention as the other two notebooks.

In [ ]:
!pip install -q -r steven/requirements-probe.txt
!python steven/src/collect_vix_yfinance.py

## Train CVAE -- HORIZON=1 (configs/cvae_h1.yaml)

Identical recipe to `configs/cvae_generative.yaml` (NLL reconstruction, momentum+VIX features, `decoder_ctx_dim=8`) -- the only thing that changed is `HORIZON` itself (now 1, not 3), which `CVAEInpainting`'s decoder width derives automatically. Writes `steven/outputs/cvae_checkpoint_h1.pt`.

Drop `--max-epochs`/`--train-windows-per-epoch` overrides below if you want a quick smoke run first instead of the full 30-epoch config.

In [ ]:
!python steven/src/train_cvae.py --config steven/configs/cvae_h1.yaml --device auto

## Train PatchTST -- HORIZON=1 (configs/patchtst_h1.yaml) -- optional

Not used by the rolling backtest (that's CVAE-only), but keeps a matching-shaped benchmark checkpoint around for `v1.md`'s PatchTST-vs-CVAE comparison, once that comparison gets redesigned for the new strategy. Skip this cell if you don't need it yet.

In [ ]:
!python steven/src/train_patchtst.py --config steven/configs/patchtst_h1.yaml --device auto

## Evaluate generative quality: diversity, calibration, context-sensitivity (optional)

Same diagnostic `colab_train_generative.ipynb` already runs (`src/evaluate_generative.py`), now HORIZON=1-safe -- confirms the retrained checkpoint isn't collapsed before trusting the backtest below. Writes `steven/outputs/generative_metrics_h1.json` + plots to `steven/outputs/generative_plots_h1/`.

In [ ]:
!python steven/src/evaluate_generative.py \
  --cvae-checkpoint steven/outputs/cvae_checkpoint_h1.pt \
  --metrics-out steven/outputs/generative_metrics_h1.json \
  --plots-dir steven/outputs/generative_plots_h1 \
  --device auto

## Run the rolling hour-by-hour backtest

The actual point of this notebook. Predicts one candle at a time over the test split, buys on an "uptrend" consensus call, holds open-endedly, exits on a divergence loss / downtrend consensus call / 1% hard stop-loss -- see `src/rolling_backtest.py`'s module docstring for the exact per-step decision order, and `rolling_hour_backtest.md` for why the consensus-gate signal specifically should be read skeptically. Writes `steven/outputs/rolling_backtest_metrics.json`.

In [ ]:
!python steven/src/rolling_trend_backtest.py \
  --cvae-checkpoint steven/outputs/cvae_checkpoint_h1.pt \
  --device auto

In [ ]:
import json

with open("steven/outputs/rolling_backtest_metrics.json") as f:
    m = json.load(f)

rb, bh = m["rolling_backtest"], m["buy_and_hold"]
print(f"n_decisions={rb['n_decisions']}  n_trades={rb['n_trades']}  forced_close_at_end={rb['forced_close_at_end']}")
print(f"total_return={rb['total_return']:+.4%}  win_rate={rb.get('win_rate')}  annual_return={rb.get('annual_return')}")
print(f"buy_and_hold total_return={bh['total_return']:+.4%}  annual_return={bh['annual_return']:+.4%}")
print("\noutcome_breakdown:")
for k, v in rb["outcome_breakdown"].items():
    print(f"  {k:24s} {v:.4f}")

## Sync results back to GitHub

Commits `steven/outputs/` (the H1 checkpoint(s), metrics JSONs, plots) from this Colab runtime and pushes to the **`steven4`** branch -- not `steven3`/`steven2`, which still hold HORIZON=3 checkpoints that this run's HORIZON=1 outputs must not overwrite.

Needs a GitHub personal access token with `repo` write scope for this push only -- entered via `getpass` below, never written to the notebook or committed anywhere.

In [ ]:
# %%bash
# git fetch origin steven4
# git merge origin/steven4 --no-edit

In [ ]:
import getpass

token = getpass.getpass("GitHub PAT (repo write, used only for this push): ")

In [ ]:
%%bash -s "$token"
TOKEN="$1"
if [ -z "$TOKEN" ]; then
  echo "Token was empty -- re-run the getpass cell above and actually paste your PAT before pressing Enter." >&2
  exit 1
fi
git config user.email "colab@ephemeral.local"
git config user.name "Colab Runtime"
git add steven/outputs
if git diff --cached --quiet; then
  echo "Nothing new to commit -- outputs unchanged from last commit."
else
  git commit -m "Train HORIZON=1 CVAE + run rolling hour-by-hour backtest from Colab run"
fi
# Push to steven4 specifically -- see the markdown note above for why.
git push "https://${TOKEN}@github.com/WoodyChang21/ECE1508_GenAI.git" HEAD:steven4

### Fallback: zip + browser download

Use this if the push cell above fails (e.g. bad token) -- zips `steven/outputs` for a manual download instead.

In [ ]:
!zip -r outputs.zip steven/outputs

try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Not in a Colab frontend session -- outputs.zip is in the working dir, grab it manually.")